# 🤖 CEM4644 · MP5 — One model for everything?
## Workshop (in class): *Façade defects, site safety, floor plans*

**No coding needed.** Each grey box is one step: click ▶, wait, read the result, answer the report question. Run from top to bottom.

One **generalist** model (Gemini) does the MP2, MP3 and MP4 tasks from words alone, and every answer is scored against the same answer keys as before. About 90 minutes.

**Before you start**
- **Gemini API key (free):** https://aistudio.google.com/apikey → in Colab, the key icon (*Secrets*) → name `GEMINI_API_KEY`, *Notebook access* on. Never paste the key into a cell.
- **GPU (optional):** *Runtime → Change runtime type → T4 GPU*, only for Step 4b.

In [ ]:
#@title ▶ Step 0 · Run me first (1–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Leave *api_key* empty to use the Colab secret. Untick *load_sam* if you have no GPU.
api_key = "" #@param {type:"string"}
model = "gemini-3.5-flash-lite" #@param ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.6-flash"]
load_sam = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp5_llm_vision", "aec_llm"
FOLDERS = ["mp5_llm_vision", "mp4_segmentation"]  # only these two lab folders are downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)     # also trims a full copy left by an earlier run
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m.split(".")[0] in (PKG, "aec_seg")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_llm import lab
lab.setup(dataset="workshop", api_key=api_key, model=model, load_sam=load_sam)


## Part 1 · Talk to the model

A vision-language model reads an image and text and answers in text. Whatever structure you want back, you ask for it in words. Watch the reply and its cost (seconds, tokens).

In [ ]:
#@title ▶ Step 1a · The examples and their answer keys { display-mode: "form" }
which = "photos" #@param ["photos", "site photos", "plans"]
lab.show_examples(which)


In [ ]:
#@title ▶ Step 1b · Ask anything about a photo { display-mode: "form" }
photo = "plain_1  (truth: plain wall (no defect))" #@param ["plain_1  (truth: plain wall (no defect))", "plain_2  (truth: plain wall (no defect))", "minor_crack_1  (truth: minor crack)", "minor_crack_2  (truth: minor crack)", "major_crack_1  (truth: major crack)", "major_crack_2  (truth: major crack)", "spalling_1  (truth: spalling)", "spalling_2  (truth: spalling)", "peeling_1  (truth: peeling paint / plaster)", "peeling_2  (truth: peeling paint / plaster)", "stain_1  (truth: stain)", "stain_2  (truth: stain)", "algae_1  (truth: algae / biological growth)", "algae_2  (truth: algae / biological growth)"]
question = "What do you see in this photo? Answer in three sentences." #@param ["What do you see in this photo? Answer in three sentences.", "Is there anything a building inspector should worry about here? Answer in two sentences.", "Describe this photo as a JSON object with the keys \"what\", \"condition\" and \"action\"."] {allow-input: true}
lab.describe(photo, question)


In [ ]:
#@title ▶ Step 1c · Getting JSON, two ways { display-mode: "form" }
#@markdown **A:** JSON asked for in the prompt. **B:** a JSON schema enforced by the API. Each run several times.
photo = "plain_1  (truth: plain wall (no defect))" #@param ["plain_1  (truth: plain wall (no defect))", "plain_2  (truth: plain wall (no defect))", "minor_crack_1  (truth: minor crack)", "minor_crack_2  (truth: minor crack)", "major_crack_1  (truth: major crack)", "major_crack_2  (truth: major crack)", "spalling_1  (truth: spalling)", "spalling_2  (truth: spalling)", "peeling_1  (truth: peeling paint / plaster)", "peeling_2  (truth: peeling paint / plaster)", "stain_1  (truth: stain)", "stain_2  (truth: stain)", "algae_1  (truth: algae / biological growth)", "algae_2  (truth: algae / biological growth)"]
repeats = 3 #@param {type:"slider", min:1, max:3, step:1}
lab.json_lab(photo, repeats)


> ### 📝 Report question 1
> From Step 1c: how many of the plain replies (A) were valid JSON, and did the label stay the same across the runs? What did the schema (B) change, and what did it not change? Why does a program that has to read the reply (to fill a table, to count, to draw a box) need B rather than A?

## Part 2 · Classification by prompt

The MP2 photos (7 classes), three prompts: the class names, the names with descriptions, the descriptions with rules. Scored against the answer key and the MP2 model.

In [ ]:
#@title ▶ Step 2a · Classify all the photos { display-mode: "form" }
prompt = "basic" #@param ["basic", "with descriptions", "with descriptions and rules"]
schema = True #@param {type:"boolean"}
show_mistakes = True #@param {type:"boolean"}
lab.classify(prompt, schema, show_mistakes)


In [ ]:
#@title ▶ Step 2b · Your own prompt { display-mode: "form" }
#@markdown Keep `{classes}` and `{intro}` in the text. Runs live on every photo (one to three minutes).
prompt_text = "{intro} Classify it into exactly one of these categories: {classes}. Reply with JSON only, no other text, in this form: {\"label\": <one category, spelled exactly as in the list>, \"confidence\": <a number from 0 to 1>, \"reason\": <one short sentence>}" #@param {type:"string"}
schema = True #@param {type:"boolean"}
lab.classify_own(prompt_text, schema)


> ### 📝 Report question 2
> From Step 2a: the accuracy of the three prompts (basic / with descriptions / with descriptions and rules) and of the MP2 model on the same 14 photos. Which classes does Gemini confuse (use the confusion table), and what did the descriptions and the rules change?

> ### 📝 Report question 3
> From Step 2b: what did you change in the prompt and what accuracy did you get? If it went up, what is the risk of tuning a prompt on the same photos you score it on (think of MP2's training / test split)?

## Part 3 · Detection and counting by prompt

The MP3 photos. Boxes come back as `[ymin, xmin, ymax, xmax]` on a 0–1000 grid and are scored like MP3: right label and an overlap of at least half (IoU ≥ 0.5).

In [ ]:
#@title ▶ Step 3a · Boxes on one photo { display-mode: "form" }
site = "site_1  (11 boxes in the answer key)" #@param ["site_1  (11 boxes in the answer key)", "site_2  (8 boxes in the answer key)", "site_3  (9 boxes in the answer key)", "site_4  (6 boxes in the answer key)", "site_5  (12 boxes in the answer key)", "site_6  (6 boxes in the answer key)"]
schema = True #@param {type:"boolean"}
lab.detect(site, schema)


In [ ]:
#@title ▶ Step 3b · All the photos, scored { display-mode: "form" }
schema = True #@param {type:"boolean"}
lab.detect_all(schema)


In [ ]:
#@title ▶ Step 3c · Just ask for the number { display-mode: "form" }
site = "site_1  (11 boxes in the answer key)" #@param ["site_1  (11 boxes in the answer key)", "site_2  (8 boxes in the answer key)", "site_3  (9 boxes in the answer key)", "site_4  (6 boxes in the answer key)", "site_5  (12 boxes in the answer key)", "site_6  (6 boxes in the answer key)"]
schema = True #@param {type:"boolean"}
lab.count(site, schema)


> ### 📝 Report question 4
> From Step 3b: Gemini's recall and precision against the MP3 YOLO model's. Which label is hardest for Gemini (helmet, NO helmet, vest, NO vest, person) and why might that be? Paste one overlay from Step 3a and explain the extras (thick boxes marked '?').

> ### 📝 Report question 5
> From Step 3c on two photos: the model's count, the count of its own boxes and the answer key. When they disagree, which one is wrong and how would you know on a site where there is no answer key? Which of the two ways of counting would you trust on a site camera, and why?

## Part 4 · Rooms on a floor plan

The MP4 plans, in square feet. Two routes: the model's own polygons, or its boxes handed to SAM 3. Every room is scored against the drawing, next to MP4's SAM 3 by phrase.

In [ ]:
#@title ▶ Step 4a · The model's own polygons { display-mode: "form" }
plan = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
mode = "LLM only (polygons)" #@param ["LLM only (polygons)"]
schema = True #@param {type:"boolean"}
lab.segment(plan, mode, schema)


In [ ]:
#@title ▶ Step 4b · The model's boxes, SAM 3's pixels { display-mode: "form" }
plan = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
mode = "LLM boxes + SAM 3" #@param ["LLM boxes + SAM 3"]
schema = True #@param {type:"boolean"}
lab.segment(plan, mode, schema)


In [ ]:
#@title ▶ Step 4c · Room by room, three ways { display-mode: "form" }
plan = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
lab.segment_compare(plan)


> ### 📝 Report question 6
> From Step 4c on one plan: copy the per-room table. Which way is closer to the drawing, the model's own polygons or its boxes handed to SAM 3, and on which rooms do they differ most? How does this compare with drawing the boxes yourself in MP4?

## Part 5 · Your image, your words

A small app for your own image and your own prompt. Needs your key.

In [ ]:
#@title ▶ Step 5 · Prompt lab { display-mode: "form" }
#@markdown Open the printed link in a new tab.
lab.prompt_app()


> ### 📝 Report question 7
> Run at least 2 experiments of your own in Step 5 (a photo from a site or from the internet, a plan, a changed prompt, the schema on and off). For each: the image, the prompt, the raw reply, and whether it was right. What kind of request broke the model, and how did it break (wrong answer, invented objects, unreadable reply)?

## Wrap-up · Generalist or specialist?

In [ ]:
#@title ▶ Step 6 · All tasks side by side { display-mode: "form" }
lab.summary()


> ### 📝 Report question 8
> From Step 6: for each task, would you use the generalist, the specialist, or both together (as in Step 4b)? Argue with the numbers you got and with what each needs: labelled data, training, a GPU, a network connection, money per request, and someone who checks. What does structured output guarantee about a reply, and what does it not guarantee?

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Photos: BD3 Building Defect Dataset (CC-BY-4.0), the test photos of MP2.
- Site photos: Roboflow 100 'construction-safety' (CC-BY-4.0), the test photos of MP3.
- Floor plans: U.S. Department of Agriculture, Miscellaneous Publication 360 (1940), public domain; prepared for MP4 (plans usda_5544, usda_5540, usda_5539).
- Model: Gemini (Google) through the Gemini API, free tier; SAM 3 (Meta, SAM License) from the MP4 folder.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp5_llm_vision`).